In [ ]:
"""
PMU FAULT DETECTION SYSTEM - COMPLETE WORKFLOW WITH FULL VISUALIZATION
Following the exact workflow: Preprocessing → Feature Preparation →
Fault Detection (TCN) → Relay Protection → CBM → Bayesian Optimization → Performance Evaluation


"""

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from scipy import signal
from scipy.stats import kurtosis, skew
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report, roc_curve, auc,
    precision_recall_curve, average_precision_score
)
from imblearn.over_sampling import SMOTE
from skopt import gp_minimize
from skopt.space import Real, Integer
from skopt.utils import use_named_args
import matplotlib.pyplot as plt
import seaborn as sns
import os
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
tf.random.set_seed(42)

# Set style for better-looking plots
# plt.style.use('seaborn-v0_8-darkgrid')
# sns.set_palette("husl")

# ============================================================================
# STEP 1: PREPROCESSING (Noise Filtering, Normalization, Windowing)
# ============================================================================

class PreprocessingModule:
    """
    Implements: Noise Filtering + Normalization + Windowing
    As shown in workflow diagram
    """
    def __init__(self, window_size=100, stride=10):
        self.window_size = window_size
        self.stride = stride
        self.scaler = RobustScaler()

    def noise_filtering(self, data):
        """Butterworth lowpass filter + Median filter for spike removal"""
        # Butterworth filter
        b, a = signal.butter(5, 0.3, btype='low')
        filtered = signal.filtfilt(b, a, data, axis=0)

        # Median filter for impulse noise
        for i in range(data.shape[1]):
            filtered[:, i] = signal.medfilt(filtered[:, i], kernel_size=3)

        return filtered

    def normalization(self, data, fit=True):
        """RobustScaler normalization"""
        if fit:
            return self.scaler.fit_transform(data)
        else:
            return self.scaler.transform(data)

    def windowing(self, data, labels):
        """Create overlapping windows with data augmentation"""
        windows, window_labels = [], []

        for i in range(0, len(data) - self.window_size + 1, self.stride):
            window = data[i:i + self.window_size]
            label = np.bincount(labels[i:i + self.window_size]).argmax()

            windows.append(window)
            window_labels.append(label)

            # Augmentation for fault cases
            if label == 1:
                # Add Gaussian noise
                noisy = window + np.random.normal(0, 0.02, window.shape)
                windows.append(noisy)
                window_labels.append(label)

                # Amplitude scaling
                scaled = window * np.random.uniform(0.95, 1.05)
                windows.append(scaled)
                window_labels.append(label)

        return np.array(windows), np.array(window_labels)

    def preprocess_pipeline(self, data, labels, fit=True):
        """Complete preprocessing pipeline"""
        # Step 1: Noise Filtering
        filtered = self.noise_filtering(data)

        # Step 2: Normalization
        normalized = self.normalization(filtered, fit=fit)

        # Step 3: Windowing
        windows, window_labels = self.windowing(normalized, labels)

        return windows, window_labels

# ============================================================================
# STEP 2: FEATURE PREPARATION (Primary Input)
# ============================================================================

class FeaturePreparation:
    """
    Prepares primary input features for TCN
    Includes SMOTE for class balancing
    """
    def __init__(self):
        pass

    def apply_smote(self, X_train, y_train):
        """Apply SMOTE for balanced training"""
        original_shape = X_train.shape
        X_train_flat = X_train.reshape(X_train.shape[0], -1)

        smote = SMOTE(random_state=42, k_neighbors=3)
        X_resampled, y_resampled = smote.fit_resample(X_train_flat, y_train)

        X_resampled = X_resampled.reshape(-1, original_shape[1], original_shape[2])

        return X_resampled, y_resampled

    def prepare_features(self, X_train, y_train, X_test, y_test):
        """Prepare features with SMOTE"""
        X_train_balanced, y_train_balanced = self.apply_smote(X_train, y_train)

        return X_train_balanced, y_train_balanced, X_test, y_test

# ============================================================================
# STEP 3: FAULT DETECTION & CLASSIFICATION (TCN Model)
# ============================================================================

class TCNFaultDetector:
    """
    Advanced TCN for fault detection
    Binary classification: Normal vs Fault
    """
    def __init__(self, input_shape, filters=128, kernel_size=3, dilations=[1,2,4,8,16]):
        self.input_shape = input_shape
        self.filters = filters
        self.kernel_size = kernel_size
        self.dilations = dilations
        self.model = None

    def residual_block(self, x, dilation_rate, dropout=0.2):
        """TCN residual block with causal convolution"""
        residual = x

        # First conv layer
        x = layers.Conv1D(self.filters, self.kernel_size, padding='causal',
                         dilation_rate=dilation_rate, activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(dropout)(x)

        # Second conv layer
        x = layers.Conv1D(self.filters, self.kernel_size, padding='causal',
                         dilation_rate=dilation_rate, activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(dropout)(x)

        # Match dimensions
        if residual.shape[-1] != self.filters:
            residual = layers.Conv1D(self.filters, 1, padding='same')(residual)

        # Residual connection
        x = layers.Add()([x, residual])
        return layers.Activation('relu')(x)

    def build_model(self):
        """Build TCN architecture"""
        inputs = layers.Input(shape=self.input_shape)

        # Initial projection
        x = layers.Conv1D(self.filters, 1)(inputs)

        for dilation in self.dilations:
            x = self.residual_block(x, dilation_rate=dilation, dropout=0.2)

        x_lstm = layers.Bidirectional(layers.LSTM(64, return_sequences=True))(x)
        x_lstm = layers.Dropout(0.15)(x_lstm)

        x = layers.Concatenate()([x, x_lstm])

        # Global pooling
        x = layers.GlobalAveragePooling1D()(x)

        # Dense classification layers
        x = layers.Dense(256, activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.15)(x)

        x = layers.Dense(128, activation='relu')(x)
        x = layers.Dropout(0.15)(x)

        # Binary output
        outputs = layers.Dense(1, activation='sigmoid')(x)

        self.model = keras.Model(inputs, outputs)

        self.model.compile(
            optimizer=keras.optimizers.AdamW(learning_rate=0.0005, weight_decay=0.0001),
            loss='binary_crossentropy',
            metrics=['accuracy', keras.metrics.AUC(name='auc'),
                    keras.metrics.Precision(name='precision'),
                    keras.metrics.Recall(name='recall')]
        )

        return self.model

    def train(self, X_train, y_train, X_val, y_val, epochs=150):
        """Train TCN with callbacks"""
        callbacks = [
            keras.callbacks.EarlyStopping(
                monitor='val_auc', patience=50, restore_best_weights=True, mode='max'
            ),
            keras.callbacks.ReduceLROnPlateau(
                monitor='val_loss', factor=0.5, patience=15, min_lr=1e-7
            )
        ]

        class_weights = {0: 1.0, 1: 1.2}

        history = self.model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=epochs,
            batch_size=32,
            callbacks=callbacks,
            class_weight=class_weights,
            verbose=1
        )

        return history

    def predict(self, X):
        """Predict fault probabilities"""
        return self.model.predict(X, verbose=0)

    def evaluate_metrics(self, y_true, y_pred_proba, threshold=0.5):
        """Evaluate fault detection metrics"""
        y_pred = (y_pred_proba >= threshold).astype(int)
        cm = confusion_matrix(y_true, y_pred)

        return {
            'accuracy': accuracy_score(y_true, y_pred),
            'precision': precision_score(y_true, y_pred, zero_division=0),
            'recall': recall_score(y_true, y_pred, zero_division=0),
            'f1_score': f1_score(y_true, y_pred, zero_division=0),
            'roc_auc': roc_auc_score(y_true, y_pred_proba),
            'confusion_matrix': cm
        }

# ============================================================================
# STEP 4: RELAY PROTECTION DECISION LOGIC
# ============================================================================

class RelayProtectionModule:
    """
    Relay protection decision logic based on fault probabilities
    Metrics: Correct Trip Rate, False Trip Rate, Missed Fault Rate, Decision Latency
    """
    def __init__(self, threshold=0.5):
        self.threshold = threshold

    def make_decision(self, fault_probability):
        """Trip decision based on threshold"""
        return 1 if fault_probability >= self.threshold else 0

    def evaluate_protection_metrics(self, y_true, y_pred_proba):
        """Calculate relay protection performance metrics"""
        import time

        decisions = []
        latencies = []

        # Simulate real-time decision making
        for prob in y_pred_proba:
            start = time.time()
            decision = self.make_decision(prob)
            latencies.append((time.time() - start) * 1000)  # Convert to ms
            decisions.append(decision)

        decisions = np.array(decisions)
        cm = confusion_matrix(y_true, decisions)
        tn, fp, fn, tp = cm.ravel()

        # Calculate rates
        total_faults = np.sum(y_true == 1)
        total_normal = np.sum(y_true == 0)

        metrics = {
            'correct_trip_rate': (tp / total_faults * 100) if total_faults > 0 else 0,
            'false_trip_rate': (fp / total_normal * 100) if total_normal > 0 else 0,
            'missed_fault_rate': (fn / total_faults * 100) if total_faults > 0 else 0,
            'avg_decision_latency_ms': np.mean(latencies),
            'max_decision_latency_ms': np.max(latencies)
        }

        return metrics

# ============================================================================
# STEP 5: CONDITION-BASED MAINTENANCE (CBM)
# ============================================================================

class CBMModule:
    """
    Condition-Based Maintenance module
    Metrics: Health Index, Maintenance Alert Accuracy, False Alarm Rate, Correlation
    """
    def __init__(self):
        self.hi_normal_threshold = 70
        self.hi_inspection_threshold = 40

    def calculate_health_index(self, fault_probability):
        """Health Index: HI = 100 * (1 - P(fault))"""
        return 100 * (1 - fault_probability)

    def classify_maintenance_action(self, health_index):
        """
        Classify maintenance action based on Health Index:
        - HI >= 70: Normal (no action)
        - 40 <= HI < 70: Inspection required
        - HI < 40: Preventive maintenance required
        """
        if health_index >= self.hi_normal_threshold:
            return 0  # Normal
        elif health_index >= self.hi_inspection_threshold:
            return 1  # Inspection
        else:
            return 2  # Preventive Maintenance

    def evaluate_cbm_metrics(self, fault_probabilities, true_labels):
        """Evaluate CBM performance metrics"""
        # Calculate Health Indices
        health_indices = [self.calculate_health_index(p) for p in fault_probabilities]

        # Classify maintenance actions
        predicted_actions = [self.classify_maintenance_action(hi) for hi in health_indices]

        # Ground truth actions (0: Normal, 2: Maintenance for faults)
        true_actions = [0 if label == 0 else 2 for label in true_labels]

        # Calculate metrics
        correct_predictions = np.sum(np.array(predicted_actions) == np.array(true_actions))
        total_predictions = len(true_actions)

        # False maintenance alarms (predicted maintenance but actually normal)
        false_alarms = np.sum((np.array(predicted_actions) > 0) & (np.array(true_actions) == 0))
        total_normal = np.sum(np.array(true_actions) == 0)

        # Correlation between HI and fault occurrence
        correlation = -np.corrcoef(health_indices, true_labels)[0, 1]

        metrics = {
            'health_index_trend_accuracy': (correct_predictions / total_predictions * 100),
            'maintenance_alert_accuracy': (correct_predictions / total_predictions * 100),
            'false_maintenance_alarm_rate': (false_alarms / total_normal * 100) if total_normal > 0 else 0,
            'avg_health_index': np.mean(health_indices),
            'health_index_std': np.std(health_indices),
            'correlation_with_fault_frequency': correlation
        }

        return metrics, health_indices

# ============================================================================
# STEP 6: BAYESIAN OPTIMIZATION
# ============================================================================

class BayesianOptimizationModule:
    """
    Bayesian Optimization for hyperparameter tuning
    Optimizes: TCN architecture, learning rate, batch size
    """
    def __init__(self, X_train, y_train, X_val, y_val):
        self.X_train = X_train
        self.y_train = y_train
        self.X_val = X_val
        self.y_val = y_val
        self.baseline_metrics = None
        self.optimized_metrics = None

    def set_baseline(self, metrics):
        """Set baseline metrics before optimization"""
        self.baseline_metrics = {
            'accuracy': metrics['accuracy'],
            'false_trip_rate': metrics.get('false_trip_rate', 0),
            'cbm_false_alarms': metrics.get('false_maintenance_alarm_rate', 0)
        }

    def objective_function(self, params):
        """Objective function for Bayesian optimization"""
        filters, lr, batch_size = params
        filters = int(filters)
        batch_size = int(batch_size)

        # Build and train model with current hyperparameters
        tcn = TCNFaultDetector(
            input_shape=(self.X_train.shape[1], self.X_train.shape[2]),
            filters=filters
        )
        tcn.build_model()

        # Recompile with new learning rate
        tcn.model.compile(
            optimizer=keras.optimizers.AdamW(learning_rate=lr, weight_decay=0.0001),
            loss='binary_crossentropy',
            metrics=['accuracy', keras.metrics.AUC(name='auc')]
        )

        # Train for fewer epochs during optimization
        history = tcn.model.fit(
            self.X_train, self.y_train,
            validation_data=(self.X_val, self.y_val),
            epochs=20,
            batch_size=batch_size,
            verbose=0
        )

        # Return negative validation AUC (we want to maximize AUC)
        best_auc = max(history.history['val_auc'])
        return -best_auc

    def optimize(self, n_calls=20):
        """Run Bayesian optimization"""
        print("\n[BAYESIAN OPTIMIZATION] Starting hyperparameter search...")

        # Define search space
        space = [
            Integer(64, 256, name='filters'),
            Real(1e-5, 1e-3, name='learning_rate', prior='log-uniform'),
            Integer(32, 128, name='batch_size')
        ]

        # Run optimization
        result = gp_minimize(
            self.objective_function,
            space,
            n_calls=n_calls,
            random_state=42,
            verbose=False
        )

        optimal_params = {
            'filters': int(result.x[0]),
            'learning_rate': result.x[1],
            'batch_size': int(result.x[2]),
            'best_auc': -result.fun
        }

        print(f"  Optimal Filters: {optimal_params['filters']}")
        print(f"  Optimal Learning Rate: {optimal_params['learning_rate']:.6f}")
        print(f"  Optimal Batch Size: {optimal_params['batch_size']}")
        print(f"  Best Validation AUC: {optimal_params['best_auc']:.4f}")

        return optimal_params

    def calculate_improvements(self, optimized_fault_metrics, optimized_relay_metrics, optimized_cbm_metrics):
        """Calculate improvement metrics"""
        if self.baseline_metrics is None:
            return None

        improvements = {
            'improvement_tcn_accuracy': (optimized_fault_metrics['accuracy'] - self.baseline_metrics['accuracy']) * 100,
            'reduction_false_trip_rate': self.baseline_metrics['false_trip_rate'] - optimized_relay_metrics['false_trip_rate'],
            'reduction_cbm_false_alarms': self.baseline_metrics['cbm_false_alarms'] - optimized_cbm_metrics['false_maintenance_alarm_rate'],
            'convergence_status': 'Optimal' if optimized_fault_metrics['accuracy'] >= 0.97 else 'In Progress'
        }

        return improvements

# ============================================================================
# STEP 7: VISUALIZATION MODULE
# ============================================================================

class VisualizationModule:
    """
    Comprehensive visualization module for PMU Fault Detection System
    Creates all required plots and exports data to CSV
    """

    def __init__(self, output_dir='pmu_results'):
        self.output_dir = output_dir
        self.timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

        # Create output directories
        os.makedirs(output_dir, exist_ok=True)
        os.makedirs(f"{output_dir}/plots", exist_ok=True)
        os.makedirs(f"{output_dir}/csv", exist_ok=True)

    def plot_training_history(self, history, save=True):
        """Plot training history: Accuracy and Loss curves"""
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))

        # Accuracy curve
        axes[0, 0].plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
        axes[0, 0].plot(history.history['val_accuracy'], label='Val Accuracy', linewidth=2)
        axes[0, 0].set_title('Model Accuracy Over Epochs', fontsize=14, fontweight='bold')
        axes[0, 0].set_xlabel('Epoch', fontsize=12)
        axes[0, 0].set_ylabel('Accuracy', fontsize=12)
        axes[0, 0].legend(loc='lower right', fontsize=10)
        axes[0, 0].grid(True, alpha=0.3)

        # Loss curve
        axes[0, 1].plot(history.history['loss'], label='Train Loss', linewidth=2)
        axes[0, 1].plot(history.history['val_loss'], label='Val Loss', linewidth=2)
        axes[0, 1].set_title('Model Loss Over Epochs', fontsize=14, fontweight='bold')
        axes[0, 1].set_xlabel('Epoch', fontsize=12)
        axes[0, 1].set_ylabel('Loss', fontsize=12)
        axes[0, 1].legend(loc='upper right', fontsize=10)
        axes[0, 1].grid(True, alpha=0.3)

        # AUC curve
        if 'auc' in history.history:
            axes[1, 0].plot(history.history['auc'], label='Train AUC', linewidth=2)
            axes[1, 0].plot(history.history['val_auc'], label='Val AUC', linewidth=2)
            axes[1, 0].set_title('Model AUC Over Epochs', fontsize=14, fontweight='bold')
            axes[1, 0].set_xlabel('Epoch', fontsize=12)
            axes[1, 0].set_ylabel('AUC', fontsize=12)
            axes[1, 0].legend(loc='lower right', fontsize=10)
            axes[1, 0].grid(True, alpha=0.3)

        # Precision curve
        if 'precision' in history.history:
            axes[1, 1].plot(history.history['precision'], label='Train Precision', linewidth=2)
            axes[1, 1].plot(history.history['val_precision'], label='Val Precision', linewidth=2)
            axes[1, 1].set_title('Model Precision Over Epochs', fontsize=14, fontweight='bold')
            axes[1, 1].set_xlabel('Epoch', fontsize=12)
            axes[1, 1].set_ylabel('Precision', fontsize=12)
            axes[1, 1].legend(loc='lower right', fontsize=10)
            axes[1, 1].grid(True, alpha=0.3)

        plt.tight_layout()

        if save:
            plt.savefig(f"{self.output_dir}/plots/training_history_{self.timestamp}.png",
                       dpi=300, bbox_inches='tight')
        plt.show()

        # Export training history to CSV
        history_df = pd.DataFrame(history.history)
        history_df['epoch'] = range(1, len(history_df) + 1)
        history_df.to_csv(f"{self.output_dir}/csv/training_history_{self.timestamp}.csv", index=False)
        print(f"✓ Training history saved to CSV")

    def plot_confusion_matrix(self, y_true, y_pred, save=True):
        """Plot confusion matrix with detailed annotations"""
        cm = confusion_matrix(y_true, y_pred)

        fig, ax = plt.subplots(figsize=(10, 8))

        # Create heatmap
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True,
                   square=True, linewidths=2, linecolor='black',
                   annot_kws={'size': 16, 'weight': 'bold'})

        # Labels
        ax.set_xlabel('Predicted Label', fontsize=14, fontweight='bold')
        ax.set_ylabel('True Label', fontsize=14, fontweight='bold')
        ax.set_title('Confusion Matrix', fontsize=16, fontweight='bold', pad=20)
        ax.set_xticklabels(['Normal (0)', 'Fault (1)'], fontsize=12)
        ax.set_yticklabels(['Normal (0)', 'Fault (1)'], fontsize=12)

        # Add performance metrics on the plot
        tn, fp, fn, tp = cm.ravel()
        accuracy = (tp + tn) / (tp + tn + fp + fn)
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

        metrics_text = f'Accuracy: {accuracy:.4f}\nPrecision: {precision:.4f}\nRecall: {recall:.4f}\nF1-Score: {f1:.4f}'
        ax.text(1.15, 0.5, metrics_text, transform=ax.transAxes,
               fontsize=11, verticalalignment='center',
               bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

        plt.tight_layout()

        if save:
            plt.savefig(f"{self.output_dir}/plots/confusion_matrix_{self.timestamp}.png",
                       dpi=300, bbox_inches='tight')
        plt.show()

        # Export confusion matrix to CSV
        cm_df = pd.DataFrame(cm,
                            columns=['Predicted_Normal', 'Predicted_Fault'],
                            index=['True_Normal', 'True_Fault'])
        cm_df.to_csv(f"{self.output_dir}/csv/confusion_matrix_{self.timestamp}.csv")
        print(f"✓ Confusion matrix saved to CSV")

    def plot_roc_curve(self, y_true, y_pred_proba, save=True):
        """Plot ROC-AUC curve"""
        fpr, tpr, thresholds = roc_curve(y_true, y_pred_proba)
        roc_auc = auc(fpr, tpr)

        fig, ax = plt.subplots(figsize=(10, 8))

        # Plot ROC curve
        ax.plot(fpr, tpr, color='darkorange', lw=3,
               label=f'ROC Curve (AUC = {roc_auc:.4f})')
        ax.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--',
               label='Random Classifier')

        # Styling
        ax.set_xlim([0.0, 1.0])
        ax.set_ylim([0.0, 1.05])
        ax.set_xlabel('False Positive Rate (FPR)', fontsize=14, fontweight='bold')
        ax.set_ylabel('True Positive Rate (TPR)', fontsize=14, fontweight='bold')
        ax.set_title('Receiver Operating Characteristic (ROC) Curve',
                    fontsize=16, fontweight='bold', pad=20)
        ax.legend(loc="lower right", fontsize=12)
        ax.grid(True, alpha=0.3)

        plt.tight_layout()

        if save:
            plt.savefig(f"{self.output_dir}/plots/roc_curve_{self.timestamp}.png",
                       dpi=300, bbox_inches='tight')
        plt.show()

        # Export ROC data to CSV
        roc_df = pd.DataFrame({
            'FPR': fpr,
            'TPR': tpr,
            'Thresholds': thresholds
        })
        roc_df.to_csv(f"{self.output_dir}/csv/roc_curve_data_{self.timestamp}.csv", index=False)
        print(f"✓ ROC curve data saved to CSV (AUC: {roc_auc:.4f})")

        return roc_auc

    def plot_precision_recall_curve(self, y_true, y_pred_proba, save=True):
        """Plot Precision-Recall curve"""
        precision, recall, thresholds = precision_recall_curve(y_true, y_pred_proba)
        avg_precision = average_precision_score(y_true, y_pred_proba)

        fig, ax = plt.subplots(figsize=(10, 8))

        # Plot PR curve
        ax.plot(recall, precision, color='blue', lw=3,
               label=f'PR Curve (AP = {avg_precision:.4f})')
        ax.axhline(y=np.sum(y_true)/len(y_true), color='red',
                  linestyle='--', lw=2, label='Baseline (No Skill)')

        # Styling
        ax.set_xlim([0.0, 1.0])
        ax.set_ylim([0.0, 1.05])
        ax.set_xlabel('Recall', fontsize=14, fontweight='bold')
        ax.set_ylabel('Precision', fontsize=14, fontweight='bold')
        ax.set_title('Precision-Recall Curve', fontsize=16, fontweight='bold', pad=20)
        ax.legend(loc="lower left", fontsize=12)
        ax.grid(True, alpha=0.3)

        plt.tight_layout()

        if save:
            plt.savefig(f"{self.output_dir}/plots/precision_recall_curve_{self.timestamp}.png",
                       dpi=300, bbox_inches='tight')
        plt.show()

        # Export PR data to CSV
        pr_df = pd.DataFrame({
            'Recall': recall,
            'Precision': precision,
            'Thresholds': np.append(thresholds, np.nan)  # thresholds is one element shorter
        })
        pr_df.to_csv(f"{self.output_dir}/csv/precision_recall_data_{self.timestamp}.csv", index=False)
        print(f"✓ Precision-Recall data saved to CSV (AP: {avg_precision:.4f})")

        return avg_precision

    def plot_fnr_fpr_analysis(self, y_true, y_pred_proba, save=True):
        """Plot FNR-FPR tradeoff analysis"""
        fpr, tpr, thresholds = roc_curve(y_true, y_pred_proba)
        fnr = 1 - tpr  # False Negative Rate = 1 - TPR

        fig, axes = plt.subplots(1, 2, figsize=(16, 6))

        # Plot 1: FNR and FPR vs Threshold
        axes[0].plot(thresholds, fpr, label='False Positive Rate', linewidth=2.5, color='red')
        axes[0].plot(thresholds, fnr, label='False Negative Rate', linewidth=2.5, color='blue')
        axes[0].set_xlabel('Classification Threshold', fontsize=14, fontweight='bold')
        axes[0].set_ylabel('Error Rate', fontsize=14, fontweight='bold')
        axes[0].set_title('FPR and FNR vs Classification Threshold',
                         fontsize=16, fontweight='bold', pad=20)
        axes[0].legend(loc='best', fontsize=12)
        axes[0].grid(True, alpha=0.3)

        # Find optimal threshold (where FPR and FNR are closest)
        optimal_idx = np.argmin(np.abs(fpr - fnr))
        optimal_threshold = thresholds[optimal_idx]
        axes[0].axvline(x=optimal_threshold, color='green', linestyle='--', linewidth=2,
                       label=f'Optimal Threshold = {optimal_threshold:.3f}')
        axes[0].legend(loc='best', fontsize=12)

        # Plot 2: FPR vs FNR tradeoff
        axes[1].plot(fpr, fnr, linewidth=3, color='purple')
        axes[1].scatter(fpr[optimal_idx], fnr[optimal_idx], s=200, c='red',
                       marker='*', zorder=5, label=f'Optimal Point (Threshold={optimal_threshold:.3f})')
        axes[1].set_xlabel('False Positive Rate (FPR)', fontsize=14, fontweight='bold')
        axes[1].set_ylabel('False Negative Rate (FNR)', fontsize=14, fontweight='bold')
        axes[1].set_title('FPR vs FNR Tradeoff', fontsize=16, fontweight='bold', pad=20)
        axes[1].legend(loc='best', fontsize=12)
        axes[1].grid(True, alpha=0.3)

        plt.tight_layout()

        if save:
            plt.savefig(f"{self.output_dir}/plots/fnr_fpr_analysis_{self.timestamp}.png",
                       dpi=300, bbox_inches='tight')
        plt.show()

        # Export FNR-FPR data to CSV
        fnr_fpr_df = pd.DataFrame({
            'Threshold': thresholds,
            'FPR': fpr,
            'FNR': fnr,
            'TPR': tpr
        })
        fnr_fpr_df.to_csv(f"{self.output_dir}/csv/fnr_fpr_analysis_{self.timestamp}.csv", index=False)
        print(f"✓ FNR-FPR analysis saved to CSV")
        print(f"  Optimal Threshold: {optimal_threshold:.4f}")

        return optimal_threshold

    def plot_health_index_distribution(self, health_indices, y_true, save=True):
        """Plot health index distribution for normal vs fault cases"""
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))

        # Separate health indices by class
        hi_normal = [health_indices[i] for i in range(len(y_true)) if y_true[i] == 0]
        hi_fault = [health_indices[i] for i in range(len(y_true)) if y_true[i] == 1]

        # Plot 1: Histogram
        axes[0].hist(hi_normal, bins=30, alpha=0.6, label='Normal', color='green', edgecolor='black')
        axes[0].hist(hi_fault, bins=30, alpha=0.6, label='Fault', color='red', edgecolor='black')
        axes[0].axvline(x=70, color='orange', linestyle='--', linewidth=2, label='Inspection Threshold')
        axes[0].axvline(x=40, color='darkred', linestyle='--', linewidth=2, label='Maintenance Threshold')
        axes[0].set_xlabel('Health Index', fontsize=14, fontweight='bold')
        axes[0].set_ylabel('Frequency', fontsize=14, fontweight='bold')
        axes[0].set_title('Health Index Distribution', fontsize=16, fontweight='bold', pad=20)
        axes[0].legend(fontsize=12)
        axes[0].grid(True, alpha=0.3)

        # Plot 2: Box plot
        box_data = [hi_normal, hi_fault]
        bp = axes[1].boxplot(box_data, labels=['Normal', 'Fault'], patch_artist=True,
                            notch=True, showmeans=True)

        # Color the box plots
        colors = ['lightgreen', 'lightcoral']
        for patch, color in zip(bp['boxes'], colors):
            patch.set_facecolor(color)

        axes[1].axhline(y=70, color='orange', linestyle='--', linewidth=2, label='Inspection Threshold')
        axes[1].axhline(y=40, color='darkred', linestyle='--', linewidth=2, label='Maintenance Threshold')
        axes[1].set_ylabel('Health Index', fontsize=14, fontweight='bold')
        axes[1].set_title('Health Index Box Plot Comparison', fontsize=16, fontweight='bold', pad=20)
        axes[1].legend(fontsize=12)
        axes[1].grid(True, alpha=0.3, axis='y')

        plt.tight_layout()

        if save:
            plt.savefig(f"{self.output_dir}/plots/health_index_distribution_{self.timestamp}.png",
                       dpi=300, bbox_inches='tight')
        plt.show()

        # Export health index data to CSV
        hi_df = pd.DataFrame({
            'Health_Index': health_indices,
            'True_Label': y_true,
            'Class': ['Normal' if y == 0 else 'Fault' for y in y_true]
        })
        hi_df.to_csv(f"{self.output_dir}/csv/health_index_data_{self.timestamp}.csv", index=False)
        print(f"✓ Health index data saved to CSV")

    def create_performance_metrics_table(self, fault_metrics, relay_metrics,
                                         cbm_metrics, optimization_metrics=None, save=True):
        """Create comprehensive performance metrics table and save to CSV"""

        # Prepare data for table
        metrics_data = []

        # 1. Fault Detection Metrics
        metrics_data.append(['FAULT DETECTION & CLASSIFICATION', '', ''])
        metrics_data.append(['Accuracy', f"{fault_metrics['accuracy']*100:.2f}%", 'TCN Model'])
        metrics_data.append(['Precision', f"{fault_metrics['precision']*100:.2f}%", 'TCN Model'])
        metrics_data.append(['Recall (Sensitivity)', f"{fault_metrics['recall']*100:.2f}%", 'TCN Model'])
        metrics_data.append(['F1-Score', f"{fault_metrics['f1_score']*100:.2f}%", 'TCN Model'])
        metrics_data.append(['ROC-AUC', f"{fault_metrics['roc_auc']*100:.2f}%", 'TCN Model'])

        # Confusion Matrix elements
        cm = fault_metrics['confusion_matrix']
        metrics_data.append(['True Negatives', f"{cm[0,0]}", 'Confusion Matrix'])
        metrics_data.append(['False Positives', f"{cm[0,1]}", 'Confusion Matrix'])
        metrics_data.append(['False Negatives', f"{cm[1,0]}", 'Confusion Matrix'])
        metrics_data.append(['True Positives', f"{cm[1,1]}", 'Confusion Matrix'])

        # 2. Relay Protection Metrics
        metrics_data.append(['', '', ''])
        metrics_data.append(['RELAY PROTECTION PERFORMANCE', '', ''])
        metrics_data.append(['Correct Trip Rate', f"{relay_metrics['correct_trip_rate']:.2f}%", 'Protection System'])
        metrics_data.append(['False Trip Rate', f"{relay_metrics['false_trip_rate']:.2f}%", 'Protection System'])
        metrics_data.append(['Missed Fault Rate', f"{relay_metrics['missed_fault_rate']:.2f}%", 'Protection System'])
        metrics_data.append(['Avg Decision Latency', f"{relay_metrics['avg_decision_latency_ms']:.4f} ms", 'Protection System'])
        metrics_data.append(['Max Decision Latency', f"{relay_metrics['max_decision_latency_ms']:.4f} ms", 'Protection System'])

        # 3. CBM Metrics
        metrics_data.append(['', '', ''])
        metrics_data.append(['CONDITION-BASED MAINTENANCE', '', ''])
        metrics_data.append(['Health Index Trend Accuracy', f"{cbm_metrics['health_index_trend_accuracy']:.2f}%", 'CBM System'])
        metrics_data.append(['Maintenance Alert Accuracy', f"{cbm_metrics['maintenance_alert_accuracy']:.2f}%", 'CBM System'])
        metrics_data.append(['False Maintenance Alarm Rate', f"{cbm_metrics['false_maintenance_alarm_rate']:.2f}%", 'CBM System'])
        metrics_data.append(['Average Health Index', f"{cbm_metrics['avg_health_index']:.2f}", 'CBM System'])
        metrics_data.append(['Health Index Std Dev', f"{cbm_metrics['health_index_std']:.2f}", 'CBM System'])
        metrics_data.append(['Correlation with Faults', f"{cbm_metrics['correlation_with_fault_frequency']:.4f}", 'CBM System'])

        # 4. Optimization Metrics (if available)
        if optimization_metrics:
            metrics_data.append(['', '', ''])
            metrics_data.append(['OPTIMIZATION EFFECTIVENESS', '', ''])
            metrics_data.append(['Improvement in TCN Accuracy', f"{optimization_metrics['improvement_tcn_accuracy']:+.2f}%", 'Bayesian Opt'])
            metrics_data.append(['Reduction in False Trip Rate', f"{optimization_metrics['reduction_false_trip_rate']:+.2f}%", 'Bayesian Opt'])
            metrics_data.append(['Reduction in CBM False Alarms', f"{optimization_metrics['reduction_cbm_false_alarms']:+.2f}%", 'Bayesian Opt'])
            metrics_data.append(['Convergence Status', optimization_metrics['convergence_status'], 'Bayesian Opt'])

        # Create DataFrame
        metrics_df = pd.DataFrame(metrics_data, columns=['Metric', 'Value', 'Component'])

        if save:
            metrics_df.to_csv(f"{self.output_dir}/csv/overall_performance_metrics_{self.timestamp}.csv",
                            index=False)
            print(f"✓ Overall performance metrics saved to CSV")

        # Display as formatted table
        print("\n" + "="*100)
        print(f"{'OVERALL PERFORMANCE METRICS TABLE':^100}")
        print("="*100)
        print(f"{'Metric':<40} {'Value':>20} {'Component':>30}")
        print("-"*100)

        for row in metrics_data:
            if row[0] == '' or 'FAULT DETECTION' in row[0] or 'RELAY PROTECTION' in row[0] or \
               'CONDITION-BASED' in row[0] or 'OPTIMIZATION' in row[0]:
                print(f"\n{row[0]:^100}")
                if row[0] != '':
                    print("-"*100)
            else:
                print(f"{row[0]:<40} {row[1]:>20} {row[2]:>30}")

        print("="*100 + "\n")

        return metrics_df

    def create_classification_report_csv(self, y_true, y_pred, save=True):
        """Create and save detailed classification report"""
        report = classification_report(y_true, y_pred,
                                      target_names=['Normal', 'Fault'],
                                      output_dict=True)

        # Convert to DataFrame
        report_df = pd.DataFrame(report).transpose()

        if save:
            report_df.to_csv(f"{self.output_dir}/csv/classification_report_{self.timestamp}.csv")
            print(f"✓ Classification report saved to CSV")

        print("\n" + "="*80)
        print("CLASSIFICATION REPORT")
        print("="*80)
        print(report_df)
        print("="*80 + "\n")

        return report_df

    def generate_all_visualizations(self, y_true, y_pred_proba, history,
                                    fault_metrics, relay_metrics, cbm_metrics,
                                    health_indices, optimization_metrics=None):
        """Generate all visualizations and export all data"""
        print("\n" + "="*100)
        print(f"{'GENERATING COMPREHENSIVE VISUALIZATIONS AND REPORTS':^100}")
        print("="*100 + "\n")

        y_pred = (y_pred_proba >= 0.5).astype(int)

        # 1. Training History
        print("[1/8] Creating training history plots...")
        self.plot_training_history(history)

        # 2. Confusion Matrix
        print("[2/8] Creating confusion matrix...")
        self.plot_confusion_matrix(y_true, y_pred)

        # 3. ROC Curve
        print("[3/8] Creating ROC curve...")
        self.plot_roc_curve(y_true, y_pred_proba)

        # 4. Precision-Recall Curve
        print("[4/8] Creating Precision-Recall curve...")
        self.plot_precision_recall_curve(y_true, y_pred_proba)

        # 5. FNR-FPR Analysis
        print("[5/8] Creating FNR-FPR analysis...")
        self.plot_fnr_fpr_analysis(y_true, y_pred_proba)

        # 6. Health Index Distribution
        print("[6/8] Creating health index distribution...")
        self.plot_health_index_distribution(health_indices, y_true)

        # 7. Performance Metrics Table
        print("[7/8] Creating overall performance metrics table...")
        self.create_performance_metrics_table(fault_metrics, relay_metrics,
                                             cbm_metrics, optimization_metrics)

        # 8. Classification Report
        print("[8/8] Creating classification report...")
        self.create_classification_report_csv(y_true, y_pred)

        print("\n" + "="*100)
        print(f"{'ALL VISUALIZATIONS AND REPORTS COMPLETED':^100}")
        print("="*100)
        print(f"\n✓ All plots saved to: {self.output_dir}/plots/")
        print(f"✓ All CSV files saved to: {self.output_dir}/csv/")
        print(f"✓ Timestamp: {self.timestamp}\n")

# ============================================================================
# STEP 8: PERFORMANCE EVALUATION
# ============================================================================

class PerformanceEvaluator:
    """
    Comprehensive performance evaluation
    Generates complete report with all metrics
    """
    def __init__(self):
        pass

    def print_comprehensive_report(self, fault_metrics, relay_metrics, cbm_metrics, optimization_metrics=None):
        """Print complete performance evaluation report"""
        print("\n" + "="*85)
        print(" "*20 + "PERFORMANCE EVALUATION REPORT")
        print("="*85)

        # 1. Fault Detection & Classification Metrics
        print("\n1. FAULT DETECTION & CLASSIFICATION METRICS (TCN)")
        print("-" * 85)
        print(f"  ACCURACY                      : {fault_metrics['accuracy']*100:>6.2f}%")
        print(f"  PRECISION                     : {fault_metrics['precision']*100:>6.2f}%")
        print(f"  RECALL (SENSITIVITY)          : {fault_metrics['recall']*100:>6.2f}%")
        print(f"  F1-SCORE                      : {fault_metrics['f1_score']*100:>6.2f}%")
        print(f"  ROC-AUC                       : {fault_metrics['roc_auc']*100:>6.2f}%")
        print(f"\n  CONFUSION MATRIX:")
        cm = fault_metrics['confusion_matrix']
        print(f"    TN: {cm[0,0]:>6}  FP: {cm[0,1]:>6}")
        print(f"    FN: {cm[1,0]:>6}  TP: {cm[1,1]:>6}")

        # 2. Relay Protection Performance Metrics
        print("\n2. RELAY PROTECTION PERFORMANCE METRICS")
        print("-" * 85)
        print(f"  CORRECT TRIP RATE (%)         : {relay_metrics['correct_trip_rate']:>6.2f}%")
        print(f"  FALSE TRIP RATE (%)           : {relay_metrics['false_trip_rate']:>6.2f}%")
        print(f"  MISSED FAULT RATE (%)         : {relay_metrics['missed_fault_rate']:>6.2f}%")
        print(f"  AVG DECISION LATENCY          : {relay_metrics['avg_decision_latency_ms']:>6.4f} ms")
        print(f"  MAX DECISION LATENCY          : {relay_metrics['max_decision_latency_ms']:>6.4f} ms")

        # 3. Condition-Based Maintenance Metrics
        print("\n3. CONDITION-BASED MAINTENANCE (CBM) METRICS")
        print("-" * 85)
        print(f"  HEALTH INDEX TREND ACCURACY   : {cbm_metrics['health_index_trend_accuracy']:>6.2f}%")
        print(f"  MAINTENANCE ALERT ACCURACY    : {cbm_metrics['maintenance_alert_accuracy']:>6.2f}%")
        print(f"  FALSE MAINTENANCE ALARM RATE  : {cbm_metrics['false_maintenance_alarm_rate']:>6.2f}%")
        print(f"  AVG HEALTH INDEX              : {cbm_metrics['avg_health_index']:>6.2f}")
        print(f"  HEALTH INDEX STD              : {cbm_metrics['health_index_std']:>6.2f}")
        print(f"  CORRELATION WITH FAULTS       : {cbm_metrics['correlation_with_fault_frequency']:>6.4f}")

        # 4. Optimization Effectiveness Metrics
        if optimization_metrics:
            print("\n4. OPTIMIZATION EFFECTIVENESS METRICS (BAYESIAN OPTIMIZATION)")
            print("-" * 85)
            print(f"  IMPROVEMENT IN TCN ACCURACY   : {optimization_metrics['improvement_tcn_accuracy']:>+6.2f}%")
            print(f"  REDUCTION IN FALSE TRIP RATE  : {optimization_metrics['reduction_false_trip_rate']:>+6.2f}%")
            print(f"  REDUCTION IN CBM FALSE ALARMS : {optimization_metrics['reduction_cbm_false_alarms']:>+6.2f}%")
            print(f"  CONVERGENCE STATUS            : {optimization_metrics['convergence_status']}")

        print("\n" + "="*85)

# ============================================================================
# MAIN PIPELINE - COMPLETE WORKFLOW WITH VISUALIZATION
# ============================================================================

class PMUFaultDetectionPipeline:
    """
    Complete PMU Fault Detection Pipeline
    Implements entire workflow from diagram with full visualization
    """
    def __init__(self):
        self.preprocessor = PreprocessingModule(window_size=100, stride=10)
        self.feature_prep = FeaturePreparation()
        self.tcn_detector = None
        self.relay_protection = RelayProtectionModule(threshold=0.5)
        self.cbm_module = CBMModule()
        self.evaluator = PerformanceEvaluator()
        self.viz = VisualizationModule(output_dir='pmu_results')

    def load_pmu_data(self, csv_path):
        """Load PMU dataset"""
        print(f"\n[PMU DATA COLLECTION] Loading: {csv_path}")
        df = pd.read_csv(csv_path)

        print(f"  Dataset Shape: {df.shape}")
        print(f"  Class Distribution: No-Fault={np.sum(df['Class_Label']==0)}, Fault={np.sum(df['Class_Label']==1)}")

        features = ['Voltage', 'Voltage_Angle', 'Current', 'Current_Angle', 'Frequency']
        return df[features].values, df['Class_Label'].values

    def run_complete_pipeline(self, csv_path, use_bayesian_opt=False):
        """Execute complete workflow"""
        print("\n" + "="*85)
        print(" "*15 + "PMU FAULT DETECTION SYSTEM - COMPLETE WORKFLOW")
        print("="*85)

        # STEP 1: Load Data
        data, labels = self.load_pmu_data(csv_path)

        # STEP 2: Preprocessing (Noise Filtering + Normalization + Windowing)
        print("\n[PREPROCESSING] Noise Filtering + Normalization + Windowing...")
        X_windows, y_windows = self.preprocessor.preprocess_pipeline(data, labels, fit=True)
        print(f"  Windows Created: {len(X_windows)}")
        print(f"  Window Distribution: No-Fault={np.sum(y_windows==0)}, Fault={np.sum(y_windows==1)}")

        # Split data
        X_train, X_test, y_train, y_test = train_test_split(
            X_windows, y_windows, test_size=0.15, random_state=42, stratify=y_windows
        )

        # STEP 3: Feature Preparation (SMOTE)
        print("\n[FEATURE PREPARATION] Applying SMOTE for Class Balance...")
        X_train, y_train, X_test, y_test = self.feature_prep.prepare_features(
            X_train, y_train, X_test, y_test
        )
        print(f"  Training Samples: {len(X_train)} (Balanced)")
        print(f"  Testing Samples: {len(X_test)}")

        # STEP 4: Bayesian Optimization (Optional)
        optimal_params = None
        if use_bayesian_opt:
            bayesian_opt = BayesianOptimizationModule(X_train, y_train, X_test, y_test)
            optimal_params = bayesian_opt.optimize(n_calls=15)

        # STEP 5: Build and Train TCN
        print("\n[FAULT DETECTION & CLASSIFICATION] Building TCN Model...")
        if optimal_params:
            self.tcn_detector = TCNFaultDetector(
                input_shape=(X_train.shape[1], X_train.shape[2]),
                filters=optimal_params['filters']
            )
        else:
            self.tcn_detector = TCNFaultDetector(
                input_shape=(X_train.shape[1], X_train.shape[2]),
                filters=128
            )

        self.tcn_detector.build_model()
        print(f"  Model Parameters: {self.tcn_detector.model.count_params():,}")

        print("\n[TRAINING] Training TCN (150 epochs with early stopping)...")
        history = self.tcn_detector.train(X_train, y_train, X_test, y_test, epochs=150)

        # STEP 6: Predictions
        print("\n[PREDICTION] Generating fault probabilities...")
        y_pred_proba = self.tcn_detector.predict(X_test).flatten()

        # STEP 7: Fault Detection Metrics
        print("\n[EVALUATION] Calculating all metrics...")
        fault_metrics = self.tcn_detector.evaluate_metrics(y_test, y_pred_proba)

        # STEP 8: Relay Protection Evaluation
        relay_metrics = self.relay_protection.evaluate_protection_metrics(y_test, y_pred_proba)

        # STEP 9: CBM Evaluation
        cbm_metrics, health_indices = self.cbm_module.evaluate_cbm_metrics(y_pred_proba, y_test)

        # STEP 10: Optimization Metrics (if Bayesian opt was used)
        optimization_metrics = None
        if use_bayesian_opt and optimal_params:
            # Set baseline from initial run
            baseline = {
                'accuracy': 0.8558,
                'false_trip_rate': 10.80,
                'cbm_false_alarms': 13.76
            }

            optimization_metrics = {
                'improvement_tcn_accuracy': (fault_metrics['accuracy'] - baseline['accuracy']) * 100,
                'reduction_false_trip_rate': baseline['false_trip_rate'] - relay_metrics['false_trip_rate'],
                'reduction_cbm_false_alarms': baseline['cbm_false_alarms'] - cbm_metrics['false_maintenance_alarm_rate'],
                'convergence_status': 'Optimal' if fault_metrics['accuracy'] >= 0.97 else 'In Progress'
            }

        # STEP 11: Performance Evaluation Report
        self.evaluator.print_comprehensive_report(
            fault_metrics, relay_metrics, cbm_metrics, optimization_metrics
        )

        # STEP 12: Generate All Visualizations
        self.viz.generate_all_visualizations(
            y_true=y_test,
            y_pred_proba=y_pred_proba,
            history=history,
            fault_metrics=fault_metrics,
            relay_metrics=relay_metrics,
            cbm_metrics=cbm_metrics,
            health_indices=health_indices,
            optimization_metrics=optimization_metrics
        )

        return {
            'fault_metrics': fault_metrics,
            'relay_metrics': relay_metrics,
            'cbm_metrics': cbm_metrics,
            'optimization_metrics': optimization_metrics,
            'history': history,
            'y_test': y_test,
            'y_pred_proba': y_pred_proba,
            'health_indices': health_indices
        }

# ============================================================================
# MAIN EXECUTION
# ============================================================================

if __name__ == "__main__":
    # Initialize pipeline
    pipeline = PMUFaultDetectionPipeline()

    # Path to your PMU dataset
    csv_path = "/content/pmu_fault_dataset.csv"

    # Run complete pipeline with full visualization
    # Set use_bayesian_opt=True to enable Bayesian optimization (slower but better results)
    results = pipeline.run_complete_pipeline(csv_path, use_bayesian_opt=False)

    # Final summary
    print("\n" + "="*85)
    print(" "*30 + "FINAL RESULTS")
    print("="*85)
    print(f"  TCN Accuracy         : {results['fault_metrics']['accuracy']*100:.2f}%")
    print(f"  F1-Score             : {results['fault_metrics']['f1_score']*100:.2f}%")
    print(f"  ROC-AUC              : {results['fault_metrics']['roc_auc']*100:.2f}%")
    print(f"\n  All plots saved to   : pmu_results/plots/")
    print(f"  All CSV files saved to: pmu_results/csv/")
    print("="*85)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    confusion_matrix, classification_report, roc_curve, auc,
    precision_recall_curve, average_precision_score
)
from datetime import datetime
import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt
plt.style.use("default")

# your plotting settings
plt.rcParams["figure.figsize"] = (12, 9)
plt.rcParams["font.family"] = "Liberation Serif"
# plt.rcParams["font.family"] = "Times New Roman"

plt.rcParams["font.size"] = 20
plt.rcParams["font.weight"] = "bold"
plt.rcParams["axes.titleweight"] = "bold"
plt.rcParams["axes.labelweight"] = "bold"
plt.rcParams["axes.grid"] = False

# Set style for better-looking plots
# plt.style.use('seaborn-v0_8-darkgrid')
# sns.set_palette("husl")

# Create output directories
output_dir = 'pmu_results'
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
os.makedirs(output_dir, exist_ok=True)
os.makedirs(f"{output_dir}/plots", exist_ok=True)
os.makedirs(f"{output_dir}/csv", exist_ok=True)

# ============================================================================
# AFTER RUNNING YOUR PIPELINE, ADD THIS CODE:
# ============================================================================
# Assuming you have these variables from your pipeline:
history = results['history']
y_test = results['y_test']
y_pred_proba = results['y_pred_proba']
fault_metrics = results['fault_metrics']
relay_metrics = results['relay_metrics']
cbm_metrics = results['cbm_metrics']
health_indices = results['health_indices']
optimization_metrics = results['optimization_metrics']

# Get predictions
y_pred = (y_pred_proba >= 0.5).astype(int)

print("\n" + "="*100)
print(f"{'GENERATING ALL PLOTS SEPARATELY':^100}")
print("="*100 + "\n")

# ============================================================================
# PLOT 1: TRAINING ACCURACY CURVE
# ============================================================================
print("[1/12] Creating Training Accuracy Curve...")

plt.figure(figsize=(12, 9))
plt.plot(history.history['accuracy'], label='Train Accuracy', linewidth=2.5, color='blue', marker='o', markersize=3)
plt.plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2.5, color='red', marker='s', markersize=3)
plt.title('Model Accuracy ', fontsize=20, fontweight='bold', pad=20)
plt.xlabel('Epoch', fontsize=20, fontweight='bold')
plt.ylabel('Accuracy', fontsize=20, fontweight='bold')
plt.legend(loc='lower right', fontsize=20, frameon=True, shadow=True)
# plt.grid(True, alpha=0.3, linestyle='--')
plt.tight_layout()
plt.savefig(f"{output_dir}/plots/1_training_accuracy1_{timestamp}.png", dpi=800, bbox_inches='tight')
plt.show()
print("✓ Training Accuracy Curve saved")

# ============================================================================
# PLOT 2: TRAINING LOSS CURVE
# ============================================================================
print("[2/12] Creating Training Loss Curve...")

plt.figure(figsize=(12, 9))
plt.plot(history.history['loss'], label='Train Loss', linewidth=2.5, color='green', marker='o', markersize=3)
plt.plot(history.history['val_loss'], label='Validation Loss', linewidth=2.5, color='orange', marker='s', markersize=3)
plt.title('Model Loss ', fontsize=20, fontweight='bold', pad=20)
plt.xlabel('Epoch', fontsize=20, fontweight='bold')
plt.ylabel('Loss', fontsize=20, fontweight='bold')
plt.legend(loc='upper right', fontsize=20, frameon=True, shadow=True)
# plt.grid(True, alpha=0.3, linestyle='--')
plt.tight_layout()
plt.savefig(f"{output_dir}/plots/2_training_loss1_{timestamp}.png", dpi=800, bbox_inches='tight')
plt.show()
print("✓ Training Loss Curve saved")

# # ============================================================================
# # PLOT 3: AUC CURVE OVER EPOCHS
# # ============================================================================
# print("[3/12] Creating AUC Curve Over Epochs...")

# if 'auc' in history.history:
#     plt.figure(figsize=(12, 7))
#     plt.plot(history.history['auc'], label='Train AUC', linewidth=2.5, color='purple', marker='o', markersize=3)
#     plt.plot(history.history['val_auc'], label='Validation AUC', linewidth=2.5, color='magenta', marker='s', markersize=3)
#     plt.title('Model AUC Over Epochs', fontsize=18, fontweight='bold', pad=20)
#     plt.xlabel('Epoch', fontsize=14, fontweight='bold')
#     plt.ylabel('AUC', fontsize=14, fontweight='bold')
#     plt.legend(loc='lower right', fontsize=12, frameon=True, shadow=True)
#     # plt.grid(True, alpha=0.3, linestyle='--')
#     plt.tight_layout()
#     plt.savefig(f"{output_dir}/plots/3_auc_curve_{timestamp}.png", dpi=300, bbox_inches='tight')
#     plt.show()
#     print("✓ AUC Curve saved")
# else:
#     print("⚠ AUC data not available in history")

# # ============================================================================
# # PLOT 4: PRECISION CURVE OVER EPOCHS
# # ============================================================================
# print("[4/12] Creating Precision Curve Over Epochs...")

# if 'precision' in history.history:
#     plt.figure(figsize=(12, 7))
#     plt.plot(history.history['precision'], label='Train Precision', linewidth=2.5, color='teal', marker='o', markersize=3)
#     plt.plot(history.history['val_precision'], label='Validation Precision', linewidth=2.5, color='coral', marker='s', markersize=3)
#     plt.title('Model Precision Over Epochs', fontsize=18, fontweight='bold', pad=20)
#     plt.xlabel('Epoch', fontsize=14, fontweight='bold')
#     plt.ylabel('Precision', fontsize=14, fontweight='bold')
#     plt.legend(loc='lower right', fontsize=12, frameon=True, shadow=True)
#     # plt.grid(True, alpha=0.3, linestyle='--')
#     plt.tight_layout()
#     plt.savefig(f"{output_dir}/plots/4_precision_curve_{timestamp}.png", dpi=300, bbox_inches='tight')
#     plt.show()
#     print("✓ Precision Curve saved")
# else:
#     print("⚠ Precision data not available in history")

# # ============================================================================
# PLOT 5: CONFUSION MATRIX
# ============================================================================
print("[5/12] Creating Confusion Matrix...")

cm = confusion_matrix(y_test, y_pred)
# plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='PuBuGn', cbar=True,
           square=True, linewidths=2, linecolor='black',
           annot_kws={'size': 18, 'weight': 'bold'})

plt.xlabel('Predicted Label', fontsize=20, fontweight='bold')
plt.ylabel('True Label', fontsize=20, fontweight='bold')
plt.title('Confusion Matrix', fontsize=20, fontweight='bold', pad=20)
plt.xticks([0.5, 1.5], ['Normal ', 'Fault '], fontsize=20)
plt.yticks([0.5, 1.5], ['Normal ', 'Fault '], fontsize=20, rotation=0)

# Add metrics annotation
tn, fp, fn, tp = cm.ravel()
accuracy = (tp + tn) / (tp + tn + fp + fn)
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

# metrics_text = f'Metrics:\nAccuracy: {accuracy:.4f}\nPrecision: {precision:.4f}\nRecall: {recall:.4f}\nF1-Score: {f1:.4f}'
# plt.text(2.5, 1, metrics_text, fontsize=11, verticalalignment='center',
#         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.7))

plt.tight_layout()
plt.savefig(f"{output_dir}/plots/5_confusion_mat1.png", dpi=800, bbox_inches='tight')
plt.show()
print("✓ Confusion Matrix saved")

# Export confusion matrix to CSV
cm_df = pd.DataFrame(cm, columns=['Predicted_Normal', 'Predicted_Fault'],
                    index=['True_Normal', 'True_Fault'])
cm_df.to_csv(f"{output_dir}/csv/confusion_matrix_{timestamp}.csv")
print("✓ Confusion Matrix CSV saved")

# ============================================================================
# PLOT 6: ROC-AUC CURVE
# ============================================================================
print("[6/12] Creating ROC-AUC Curve...")

fpr, tpr, thresholds_roc = roc_curve(y_test, y_pred_proba)
roc_auc = auc(fpr, tpr)

# plt.figure(figsize=(10, 8))
plt.plot(fpr, tpr, color='darkorange', lw=3, label=f'ROC Curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier ')
# plt.fill_between(fpr, tpr, alpha=0.2, color='orange')

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (FPR)', fontsize=20, fontweight='bold')
plt.ylabel('True Positive Rate (TPR)', fontsize=20, fontweight='bold')
plt.title('ROC Curve', fontsize=20, fontweight='bold', pad=20)
plt.legend(loc="lower right", fontsize=20, frameon=True, shadow=True)
# plt.grid(True, alpha=0.3, linestyle='--')
plt.tight_layout()
plt.savefig(f"{output_dir}/plots/6_roc_curve1.png", dpi=800, bbox_inches='tight')
plt.show()
print(f"✓ ROC Curve saved (AUC: {roc_auc:.4f})")

# Export ROC data to CSV
roc_df = pd.DataFrame({'FPR': fpr, 'TPR': tpr, 'Thresholds': thresholds_roc})
roc_df.to_csv(f"{output_dir}/csv/roc_curve_data_{timestamp}.csv", index=False)
print("✓ ROC Curve data CSV saved")

# ============================================================================
# PLOT 7: PRECISION-RECALL CURVE
# ============================================================================
print("[7/12] Creating Precision-Recall Curve...")

precision_vals, recall_vals, thresholds_pr = precision_recall_curve(y_test, y_pred_proba)
avg_precision = average_precision_score(y_test, y_pred_proba)

# plt.figure(figsize=(10, 8))
plt.plot(recall_vals, precision_vals, color='blue', lw=3, label=f'PR Curve (AP = {avg_precision:.4f})')
# plt.axhline(y=np.sum(y_test)/len(y_test), color='red', linestyle='--', lw=2,
#            label=f'Baseline (No Skill = {np.sum(y_test)/len(y_test):.4f})')
# plt.fill_between(recall_vals, precision_vals, alpha=0.2, color='blue')

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Recall', fontsize=20, fontweight='bold')
plt.ylabel('Precision', fontsize=20, fontweight='bold')
plt.title('Precision-Recall Curve', fontsize=20, fontweight='bold', pad=20)
plt.legend(loc="lower left", fontsize=20, frameon=True, shadow=True)
# plt.grid(True, alpha=0.3, linestyle='--')
plt.tight_layout()
plt.savefig(f"{output_dir}/plots/7_precision_recall_curve1.png", dpi=800, bbox_inches='tight')
plt.show()
print(f"✓ Precision-Recall Curve saved (AP: {avg_precision:.4f})")

# Export PR data to CSV
pr_df = pd.DataFrame({
    'Recall': recall_vals,
    'Precision': precision_vals,
    'Thresholds': np.append(thresholds_pr, np.nan)
})
pr_df.to_csv(f"{output_dir}/csv/precision_recall_data_{timestamp}.csv", index=False)
print("✓ Precision-Recall data CSV saved")

# # ============================================================================
# # PLOT 8: FNR AND FPR VS THRESHOLD
# # ============================================================================
# print("[8/12] Creating FNR and FPR vs Threshold Plot...")

# fpr_analysis, tpr_analysis, thresholds_analysis = roc_curve(y_test, y_pred_proba)
# fnr = 1 - tpr_analysis

# plt.figure(figsize=(12, 7))
# plt.plot(thresholds_analysis, fpr_analysis, label='False Positive Rate (FPR)',
#         linewidth=3, color='red', marker='o', markersize=4, markevery=10)
# plt.plot(thresholds_analysis, fnr, label='False Negative Rate (FNR)',
#         linewidth=3, color='blue', marker='s', markersize=4, markevery=10)

# # Find optimal threshold
# optimal_idx = np.argmin(np.abs(fpr_analysis - fnr))
# optimal_threshold = thresholds_analysis[optimal_idx]
# plt.axvline(x=optimal_threshold, color='green', linestyle='--', linewidth=2.5,
#            label=f'Optimal Threshold = {optimal_threshold:.3f}')
# plt.scatter(optimal_threshold, fpr_analysis[optimal_idx], s=200, c='green',
#            marker='*', zorder=5, edgecolors='black', linewidths=2)

# plt.xlabel('Classification Threshold', fontsize=14, fontweight='bold')
# plt.ylabel('Error Rate', fontsize=14, fontweight='bold')
# plt.title('False Positive Rate and False Negative Rate vs Threshold',
#          fontsize=18, fontweight='bold', pad=20)
# plt.legend(loc='best', fontsize=12, frameon=True, shadow=True)
# plt.grid(True, alpha=0.3, linestyle='--')
# plt.tight_layout()
# plt.savefig(f"{output_dir}/plots/8_fnr_fpr_vs_threshold_{timestamp}.png", dpi=300, bbox_inches='tight')
# plt.show()
# print(f"✓ FNR-FPR vs Threshold saved (Optimal Threshold: {optimal_threshold:.4f})")

# # ============================================================================
# # PLOT 9: FPR VS FNR TRADEOFF
# # ============================================================================
# print("[9/12] Creating FPR vs FNR Tradeoff Plot...")

# plt.figure(figsize=(10, 8))
# plt.plot(fpr_analysis, fnr, linewidth=3, color='purple', marker='o', markersize=4, markevery=20)
# plt.scatter(fpr_analysis[optimal_idx], fnr[optimal_idx], s=300, c='red',
#            marker='*', zorder=5, edgecolors='black', linewidths=2,
#            label=f'Optimal Point (Threshold={optimal_threshold:.3f})')

# plt.xlabel('False Positive Rate (FPR)', fontsize=14, fontweight='bold')
# plt.ylabel('False Negative Rate (FNR)', fontsize=14, fontweight='bold')
# plt.title('FPR vs FNR Tradeoff', fontsize=18, fontweight='bold', pad=20)
# plt.legend(loc='best', fontsize=12, frameon=True, shadow=True)
# plt.grid(True, alpha=0.3, linestyle='--')
# plt.tight_layout()
# plt.savefig(f"{output_dir}/plots/9_fpr_vs_fnr_tradeoff_{timestamp}.png", dpi=300, bbox_inches='tight')
# plt.show()
# print("✓ FPR vs FNR Tradeoff saved")

# # Export FNR-FPR data to CSV
# fnr_fpr_df = pd.DataFrame({
#     'Threshold': thresholds_analysis,
#     'FPR': fpr_analysis,
#     'FNR': fnr,
#     'TPR': tpr_analysis
# })
# fnr_fpr_df.to_csv(f"{output_dir}/csv/fnr_fpr_analysis_{timestamp}.csv", index=False)
# print("✓ FNR-FPR analysis CSV saved")

# ============================================================================
# PLOT 10: HEALTH INDEX HISTOGRAM
# ============================================================================
print("[10/12] Creating Health Index Histogram...")

hi_normal = [health_indices[i] for i in range(len(y_test)) if y_test[i] == 0]
hi_fault = [health_indices[i] for i in range(len(y_test)) if y_test[i] == 1]

plt.figure(figsize=(12, 9))
plt.hist(hi_normal, bins=40, alpha=0.6, label='Normal', color='green', edgecolor='black', linewidth=1.2)
plt.hist(hi_fault, bins=40, alpha=0.6, label='Fault', color='red', edgecolor='black', linewidth=1.2)
plt.axvline(x=70, color='orange', linestyle='--', linewidth=2.5, label='Inspection Threshold (70)')
plt.axvline(x=40, color='darkred', linestyle='--', linewidth=2.5, label='Maintenance Threshold (40)')

plt.xlabel('Health Index', fontsize=20, fontweight='bold')
plt.ylabel('Frequency', fontsize=20, fontweight='bold')
plt.title('Health Index Distribution: Normal vs Fault', fontsize=20, fontweight='bold', pad=20)
plt.legend(fontsize=20, frameon=True, shadow=True)

plt.tight_layout()
plt.savefig(f"{output_dir}/plots/10_health_index_histogram_{timestamp}.png", dpi=300, bbox_inches='tight')
plt.show()
print("✓ Health Index Histogram saved")

# ============================================================================
# PLOT 11: HEALTH INDEX BOX PLOT
# ============================================================================
print("[11/12] Creating Health Index Box Plot...")

# plt.figure(figsize=(10, 8))
box_data = [hi_normal, hi_fault]
bp = plt.boxplot(box_data, labels=['Normal', 'Fault'], patch_artist=True,
                notch=True, showmeans=True, meanline=True,
                boxprops=dict(linewidth=2),
                whiskerprops=dict(linewidth=2),
                capprops=dict(linewidth=2),
                medianprops=dict(linewidth=2.5, color='darkblue'),
                meanprops=dict(linewidth=2.5, color='darkgreen'))

# Color the boxes
colors = ['lightgreen', 'lightcoral']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

plt.axhline(y=70, color='orange', linestyle='--', linewidth=2.5, label='Inspection Threshold (70)')
plt.axhline(y=40, color='darkred', linestyle='--', linewidth=2.5, label='Maintenance Threshold (40)')

plt.ylabel('Health Index', fontsize=20, fontweight='bold')
plt.xlabel('Class', fontsize=20, fontweight='bold')
plt.title('Health Index Box Plot: Normal vs Fault', fontsize=20, fontweight='bold', pad=20)
plt.legend(loc='upper right',fontsize=20, frameon=True, shadow=True)

plt.tight_layout()
plt.savefig(f"{output_dir}/plots/11_health_index_boxplot_{timestamp}.png", dpi=300, bbox_inches='tight')
plt.show()
print("✓ Health Index Box Plot saved")

# Export health index data to CSV
hi_df = pd.DataFrame({
    'Health_Index': health_indices,
    'True_Label': y_test,
    'Class': ['Normal' if y == 0 else 'Fault' for y in y_test]
})
hi_df.to_csv(f"{output_dir}/csv/health_index_data_{timestamp}.csv", index=False)
print("✓ Health Index data CSV saved")

# ============================================================================
# PLOT 12: OVERALL PERFORMANCE METRICS BAR CHART
# ============================================================================
print("[12/12] Creating Overall Performance Metrics Bar Chart...")

# metrics_names = [
#     'Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC',
#     'Correct Trip\nRate', 'False Trip\nRate', 'Health Index\nAccuracy'
# ]
metrics_names = [
    'Accuracy', 'Precision', 'Recall', 'F1-Score'
]
metrics_values = [
    fault_metrics['accuracy'],
    fault_metrics['precision'],
    fault_metrics['recall'],
    fault_metrics['f1_score'],
    # fault_metrics['roc_auc'] * 100,
    # relay_metrics['correct_trip_rate'],
    # relay_metrics['false_trip_rate'],
    # cbm_metrics['health_index_trend_accuracy']
]

# colors_bar = ['#2ecc71', '#3498db', '#9b59b6', '#e74c3c', '#f39c12',
#               '#1abc9c', '#e67e22', '#34495e']
colors_bar = ['#2ecc71', '#3498db', '#9b59b6', '#e74c3c']
plt.figure(figsize=(14, 8))
bars = plt.bar(metrics_names, metrics_values, color=colors_bar, alpha=0.8,
              edgecolor='black', linewidth=1.5)

# Add value labels on bars
for i, (bar, value) in enumerate(zip(bars, metrics_values)):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
            f'{value:.4f}',
            ha='center', va='bottom', fontsize=18, fontweight='bold')

plt.ylabel('Values', fontsize=20, fontweight='bold')
plt.xlabel('Metrics', fontsize=20, fontweight='bold')
plt.title('Performance Metrics ', fontsize=20, fontweight='bold', pad=20)
plt.ylim([0, 1.1])

plt.xticks(rotation=0, ha='right', fontsize=20)
plt.tight_layout()
plt.savefig(f"{output_dir}/plots/12_overall_metrics_bar_{timestamp}.png", dpi=800, bbox_inches='tight')
plt.show()
print("✓ Overall Performance Metrics Bar Chart saved")

# ============================================================================
# EXPORT TRAINING HISTORY TO CSV
# ============================================================================
print("\n[BONUS] Exporting Training History to CSV...")

history_df = pd.DataFrame(history.history)
history_df['epoch'] = range(1, len(history_df) + 1)
history_df.to_csv(f"{output_dir}/csv/training_history_{timestamp}.csv", index=False)
print("✓ Training History CSV saved")

# ============================================================================
# EXPORT CLASSIFICATION REPORT TO CSV
# ============================================================================
print("[BONUS] Exporting Classification Report to CSV...")

report = classification_report(y_test, y_pred, target_names=['Normal', 'Fault'], output_dict=True)
report_df = pd.DataFrame(report).transpose()
report_df.to_csv(f"{output_dir}/csv/classification_report_{timestamp}.csv")
print("✓ Classification Report CSV saved")


# FINAL SUMMARY
# ============================================================================
print("\n" + "="*100)
print(f"{'ALL PLOTS AND CSV FILES GENERATED SUCCESSFULLY':^100}")
print("="*100)
print(f"\n📊 Total Plots Generated: 12")
print(f"📁 Total CSV Files Generated: 8")
print(f"\n📂 Plots Location: {output_dir}/plots/")
print(f"📂 CSV Location: {output_dir}/csv/")
print(f"🕒 Timestamp: {timestamp}")
print("\n" + "="*100)

In [ ]:
# ======================================================================
# OVERALL FPR vs FNR BAR PLOT (RAW RATE VALUES, NOT %)
# ======================================================================
print("[UPDATED] Creating Overall FPR vs FNR Bar Plot (Raw Rates)...")

tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

fpr_overall = fp / (fp + tn) if (fp + tn) > 0 else 0.0
fnr_overall = fn / (fn + tp) if (fn + tp) > 0 else 0.0

# plt.figure(figsize=(8, 6))
bars = plt.bar(
    ['False Positive Rate (FPR)', 'False Negative Rate (FNR)'],
    [fpr_overall, fnr_overall],
    edgecolor='black',
    linewidth=2,
    color=['#C8AAAA', '#FFAAB8'],
    alpha=0.8
)

# Annotate raw values|
for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        height,
        f'{height:.3f}',
        ha='center',
        va='bottom',
        fontsize=20,
        fontweight='bold'
    )

plt.ylabel('Error Rate', fontsize=20, fontweight='bold')
plt.title('FPR vs FNR', fontsize=20, fontweight='bold', pad=15)
plt.xlabel('Error Type', fontsize=20, fontweight='bold')
plt.xticks(fontsize=20)
plt.ylim(0, max(fpr_overall, fnr_overall) * 1.2 + 0.01)
# plt.grid(axis='y', alpha=0.3, linestyle='--')

plt.tight_layout()
plt.savefig(f"{output_dir}/plots/fpr_fnr_raw_{timestamp}.png", dpi=300, bbox_inches='tight')
plt.show()

print("✓ Raw FPR–FNR Bar Plot saved")


In [1]:
!pip install scikit-optimize

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.8/107.8 kB 9.5 MB/s eta 0:00:00
